# Fraud Shield - Exploratory Data Analysis

Goal: understand class balance, feature distributions, and fraud patterns
in the Kaggle credit card transactions dataset before feature engineering.

**Inputs:** `data/raw/fraudTrain.csv`, `data/raw/fraudTest.csv`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)


## 1. Load data


In [ ]:
train = pd.read_csv('../data/raw/fraudTrain.csv')
test = pd.read_csv('../data/raw/fraudTest.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
train.head()


In [ ]:
train.info()


## 2. Missing values


In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('Columns with missing values:' if len(missing) else 'No missing values found.')
missing


## 3. Class balance

`is_fraud` is expected to be heavily imbalanced -- this drives the
SMOTE/undersampling/class-weighting decision in the next notebook.


In [ ]:
fraud_counts = train['is_fraud'].value_counts()
fraud_pct = train['is_fraud'].value_counts(normalize=True) * 100

print(fraud_counts)
print()
print(fraud_pct.round(4))

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x='is_fraud', data=train, ax=ax)
ax.set_title('Class balance: legit (0) vs fraud (1)')
ax.set_yscale('log')
plt.show()


## 4. Transaction amount (`amt`)

Compare the distribution of transaction amounts for legit vs fraudulent
transactions -- fraud is often associated with unusual amount patterns.


In [ ]:
print(train.groupby('is_fraud')['amt'].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(train[train['is_fraud'] == 0]['amt'], bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Amount distribution - legit')
axes[0].set_xlim(0, 500)

sns.histplot(train[train['is_fraud'] == 1]['amt'], bins=50, ax=axes[1], color='indianred')
axes[1].set_title('Amount distribution - fraud')
axes[1].set_xlim(0, 1500)

plt.tight_layout()
plt.show()


## 5. Time patterns

Parse `trans_date_trans_time` and look at fraud rate by hour of day
and day of week -- useful signal for the velocity features later.


In [ ]:
train['trans_date_trans_time'] = pd.to_datetime(train['trans_date_trans_time'])
train['hour'] = train['trans_date_trans_time'].dt.hour
train['day_of_week'] = train['trans_date_trans_time'].dt.day_name()

fraud_by_hour = train.groupby('hour')['is_fraud'].mean() * 100

fig, ax = plt.subplots(figsize=(10, 4))
fraud_by_hour.plot(kind='bar', ax=ax, color='darkorange')
ax.set_title('Fraud rate (%) by hour of day')
ax.set_ylabel('Fraud rate (%)')
plt.show()


In [ ]:
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
fraud_by_dow = train.groupby('day_of_week')['is_fraud'].mean().reindex(dow_order) * 100

fig, ax = plt.subplots(figsize=(8, 4))
fraud_by_dow.plot(kind='bar', ax=ax, color='seagreen')
ax.set_title('Fraud rate (%) by day of week')
ax.set_ylabel('Fraud rate (%)')
plt.show()


## 6. Merchant category

Some categories (e.g. online shopping, misc_net) tend to see higher fraud rates.


In [ ]:
fraud_by_category = (train.groupby('category')['is_fraud'].mean() * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
fraud_by_category.plot(kind='barh', ax=ax, color='mediumpurple')
ax.set_title('Fraud rate (%) by merchant category')
ax.set_xlabel('Fraud rate (%)')
plt.tight_layout()
plt.show()


## 7. Geo-distance sanity check

Preview of the merchant/cardholder distance feature that will be formalized
in `src/features/engineering.py`. Fraudulent transactions often occur
farther from the cardholder's usual location.


In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    r = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))

train['geo_distance_km'] = haversine_distance(
    train['lat'], train['long'], train['merch_lat'], train['merch_long']
)

print(train.groupby('is_fraud')['geo_distance_km'].describe())

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(x='is_fraud', y='geo_distance_km', data=train, ax=ax, showfliers=False)
ax.set_title('Merchant-cardholder distance: legit vs fraud')
plt.show()


## 8. City population (`city_pop`)

Check whether fraud correlates with cardholder location density
(rural vs urban).


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(x='is_fraud', y='city_pop', data=train, ax=ax, showfliers=False)
ax.set_title('City population: legit vs fraud')
plt.show()


## 9. Correlation among numeric features


In [ ]:
numeric_cols = ['amt', 'city_pop', 'lat', 'long', 'merch_lat', 'merch_long',
                'geo_distance_km', 'is_fraud']
corr = train[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation matrix')
plt.tight_layout()
plt.show()


## 10. Key takeaways

Fill in after running against the real data, e.g.:

- Class imbalance ratio: legit vs fraud (~X% fraud)
- Fraud skews toward [amount range / time of day / category]
- `geo_distance_km` shows [stronger / weaker] separation than expected -- 
  informs whether it's a high-value feature for the supervised models
- No / some missing values requiring [imputation strategy]

**Next notebook:** `02_feature_engineering.ipynb` -- formalize velocity,
geo-distance, and spending-profile features, then push to SageMaker Feature Store.
